In [1]:
from pathlib import Path
import os
# The project folder is the folder that contains data/, notebooks/, results/ ...
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', PROJECT_DIR))
print('Project folder:', PROJECT_DIR)
assert (PROJECT_DIR / 'data').exists(), 'Project folder not found - open this notebook from the notebooks folder of the project (see your Steps file)'


Project folder: /Users/pranav/Documents/game-chat-toxicity-ml 


In [2]:
import re, html, json
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
 
USE_SAMPLE = False          # True = tiny practice data, False = the real labelled data
SEED = 42
LAB_DIR, RES_DIR, MODEL_DIR = PROJECT_DIR / 'data' / 'labelled', PROJECT_DIR / 'results', PROJECT_DIR / 'models'
(RES_DIR / 'preds').mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(exist_ok=True)
 
prefix = 'sample_' if USE_SAMPLE else ''
train = pd.read_csv(LAB_DIR / f'{prefix}train.csv'); test = pd.read_csv(LAB_DIR / f'{prefix}test.csv')
print('train', len(train), '| test', len(test))
print(train.label.value_counts().sort_index().to_dict())


train 1583 | test 396
{0: 886, 1: 584, 2: 113}


In [3]:
def clean_text(s):
    s = html.unescape(str(s)).lower()
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"[\u2019'`]", "", s)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    s = re.sub(r"[^a-z0-9?! ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()
 
TOKEN = r"(?u)\b\w+\b|[?!]"
for d in (train, test): d['clean'] = d.text.map(clean_text)
y_train, y_test = train.label.values, test.label.values
 
MIN_DF = 1 if USE_SAMPLE else 2
tfidf = TfidfVectorizer(token_pattern=TOKEN, ngram_range=(1, 2), min_df=MIN_DF, max_features=3000, sublinear_tf=True)
X_train = tfidf.fit_transform(train.clean); X_test = tfidf.transform(test.clean)
print('TF-IDF matrix:', X_train.shape)



TF-IDF matrix: (1583, 1179)


In [4]:
CLASSES = [0, 1, 2]
results = []
def record(name, y_pred, task='3-class', features='TF-IDF', y_true=None, ids=None):
    y_true = y_test if y_true is None else y_true
    ids = test.id.values if ids is None else ids
    labels = CLASSES if task == '3-class' else [0, 1]
    f1s = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    row = {'model': name, 'family': 'TF-IDF', 'features': features, 'task': task,
           'accuracy': round(accuracy_score(y_true, y_pred), 4),
           'macro_f1': round(float(np.mean(f1s)), 4)}
    for i, f in enumerate(f1s): row[f'f1_class{i}'] = round(float(f), 4)
    results.append(row)
    slug = re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')
    pd.DataFrame({'id': ids, 'y_true': y_true, 'y_pred': y_pred}).to_csv(RES_DIR / 'preds' / f'{slug}.csv', index=False)
    print(f"{name:45s} acc={row['accuracy']:.4f}  macroF1={row['macro_f1']:.4f}  F1/class={[round(float(f),2) for f in f1s]}")



In [5]:
yb_train, yb_test = (y_train > 0).astype(int), (y_test > 0).astype(int)
for kernel in ['linear', 'rbf', 'poly']:
    svm = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=SEED).fit(X_train, yb_train)
    record(f'Binary SVM ({kernel} kernel)', svm.predict(X_test), task='binary', y_true=yb_test)


Binary SVM (linear kernel)                    acc=0.9217  macroF1=0.9193  F1/class=[0.93, 0.91]
Binary SVM (rbf kernel)                       acc=0.8914  macroF1=0.8871  F1/class=[0.91, 0.87]
Binary SVM (poly kernel)                      acc=0.7879  macroF1=0.7694  F1/class=[0.83, 0.7]


In [6]:
cv_all = CountVectorizer(token_pattern=TOKEN, ngram_range=(1, 2), min_df=MIN_DF, binary=True)
Xc = cv_all.fit_transform(train.clean)
toxic = (y_train > 0)
df_tox = np.asarray(Xc[toxic].sum(0)).ravel(); df_neu = np.asarray(Xc[~toxic].sum(0)).ravel()
lift = ((df_tox + 1) / (toxic.sum() + 2)) / ((df_neu + 1) / ((~toxic).sum() + 2))
terms = np.array(cv_all.get_feature_names_out())
KEEP = {'u', 'ur', 'you', 'your', 'youre', 'ya'}                      # short words that matter in insults
def only_filler(t):                                                  # e.g. 'of', 'this is' -> not a toxic phrase
    return all((w in ENGLISH_STOP_WORDS) and (w not in KEEP) for w in t.split())
MIN_SUPPORT = 2 if USE_SAMPLE else 3                                 # phrase must appear in >= 3 toxic messages
ok = (df_tox >= MIN_SUPPORT) & (lift > 1.5) & np.array([not only_filler(t) for t in terms])
score = df_tox * ok
order = np.argsort(-score)[:165]
TOXIC_DICT = [terms[i] for i in order if score[i] > 0]
print(len(TOXIC_DICT), 'dictionary phrases, e.g.', TOXIC_DICT[:25])
 
cv_dict = CountVectorizer(token_pattern=TOKEN, ngram_range=(1, 2), vocabulary=TOXIC_DICT)
nb_dict = MultinomialNB(alpha=1.0).fit(cv_dict.transform(train.clean), y_train)
record('Naive Bayes (165-phrase dictionary)', nb_dict.predict(cv_dict.transform(test.clean)), features='dictionary counts')
 
nb_tfidf = MultinomialNB(alpha=0.1).fit(X_train, y_train)
record('Naive Bayes (TF-IDF)', nb_tfidf.predict(X_test))


153 dictionary phrases, e.g. ['report', 'fucking', 'ez', 'fuck', 'shit', 'bot', 'noob', 'idiot', 'team', 'suck', 'retard', 'reported', 'bad', 'ur', 'retarded', 'bitch', 'cancer', 'trash', 'stupid', 'fuckin', 'shut', 'stfu', 'fucked', 'afk', 'troll']
Naive Bayes (165-phrase dictionary)           acc=0.8662  macroF1=0.7075  F1/class=[0.9, 0.87, 0.35]
Naive Bayes (TF-IDF)                          acc=0.8308  macroF1=0.7513  F1/class=[0.87, 0.8, 0.58]


In [7]:
knn_models = {}
for k in [1, 2, 3, 6]:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', algorithm='brute').fit(X_train, y_train)
    knn_models[k] = knn
    record(f'KNN (K={k})', knn.predict(X_test))


KNN (K=1)                                     acc=0.6894  macroF1=0.5846  F1/class=[0.77, 0.56, 0.42]
KNN (K=2)                                     acc=0.6667  macroF1=0.4965  F1/class=[0.77, 0.43, 0.29]
KNN (K=3)                                     acc=0.6566  macroF1=0.4938  F1/class=[0.76, 0.45, 0.28]
KNN (K=6)                                     acc=0.6212  macroF1=0.3798  F1/class=[0.75, 0.26, 0.13]


In [8]:
rf_models = {}
for name, kw in [('Random Forest (Gini)', dict(criterion='gini')),
                 ('Random Forest (Entropy)', dict(criterion='entropy')),
                 ('Random Forest (Entropy, max depth=32)', dict(criterion='entropy', max_depth=32))]:
    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1, **kw).fit(X_train, y_train)
    rf_models[name] = rf
    record(name, rf.predict(X_test))


Random Forest (Gini)                          acc=0.9015  macroF1=0.8247  F1/class=[0.93, 0.89, 0.65]
Random Forest (Entropy)                       acc=0.8990  macroF1=0.8226  F1/class=[0.93, 0.89, 0.65]
Random Forest (Entropy, max depth=32)         acc=0.8460  macroF1=0.7788  F1/class=[0.89, 0.8, 0.65]


In [9]:
res = pd.DataFrame(results)
display(res)
if not USE_SAMPLE:
    res.to_csv(RES_DIR / 'tfidf_results.csv', index=False)
 
three = res[res.task == '3-class'].sort_values('macro_f1', ascending=False)
best_name = three.iloc[0].model
print('Best 3-class model by macro-F1:', best_name)
candidates = {'Naive Bayes (TF-IDF)': nb_tfidf, **{f'KNN (K={k})': m for k, m in knn_models.items()}, **rf_models}
if not USE_SAMPLE:
    joblib.dump(tfidf, MODEL_DIR / 'tfidf_vectorizer.joblib')
    joblib.dump(candidates[best_name], MODEL_DIR / 'best_tfidf_model.joblib', compress=3)
    json.dump({'name': best_name}, open(MODEL_DIR / 'best_tfidf_meta.json', 'w'))
    print('saved vectorizer + best model to', MODEL_DIR)


,model,family,features,task,accuracy,macro_f1,f1_class0,f1_class1,f1_class2
0,Binary SVM (linear kernel),TF-IDF,TF-IDF,binary,0.9217,0.9193,0.9333,0.9052,NaN
1,Binary SVM (rbf kernel),TF-IDF,TF-IDF,binary,0.8914,0.8871,0.9091,0.8652,NaN
2,Binary SVM (poly kernel),TF-IDF,TF-IDF,binary,0.7879,0.7694,0.8346,0.7042,NaN
3,Naive Bayes (165-phrase dictionary),TF-IDF,dictionary counts,3-class,0.8662,0.7075,0.9008,0.8686,0.3529
4,Naive Bayes (TF-IDF),TF-IDF,TF-IDF,3-class,0.8308,0.7513,0.8747,0.8014,0.5778
5,KNN (K=1),TF-IDF,TF-IDF,3-class,0.6894,0.5846,0.7732,0.5620,0.4186
6,KNN (K=2),TF-IDF,TF-IDF,3-class,0.6667,0.4965,0.7709,0.4330,0.2857
7,KNN (K=3),TF-IDF,TF-IDF,3-class,0.6566,0.4938,0.7581,0.4455,0.2778
8,KNN (K=6),TF-IDF,TF-IDF,3-class,0.6212,0.3798,0.7500,0.2604,0.1290
9,Random Forest (Gini),TF-IDF,TF-IDF,3-class,0.9015,0.8247,0.9293,0.8936,0.6512


Best 3-class model by macro-F1: Random Forest (Gini)
saved vectorizer + best model to /Users/pranav/Documents/game-chat-toxicity-ml /models


In [10]:
best_pred = candidates[best_name].predict(X_test)
print(pd.DataFrame(confusion_matrix(y_test, best_pred, labels=CLASSES),
                   index=['true 0', 'true 1', 'true 2'], columns=['pred 0', 'pred 1', 'pred 2']))


        pred 0  pred 1  pred 2
true 0     217       5       0
true 1      19     126       1
true 2       9       5      14
